# Boxes → `tie` outlines with Segment Anything (task 6)

Runs in Colab (**Runtime → Change runtime type → T4 GPU**, about 5 min for 301 images).

1. Downloads the forked **stirrup-recognition-dataset** (CC BY 4.0, by jiawens Workspace) from Diego's Roboflow workspace: one box per loose stirrup.
2. Prompts **SAM 2.1** (Apache-2.0) with each box and keeps the stirrup's outline. A stirrup is a closed loop, so the hole in the middle is joined to the outer edge by a thin cut (a "keyhole"): the inside of the loop stays unlabelled.
3. Drops masks that look wrong (almost empty, or filling the whole box), shows a grid to check by eye, and uploads the rest to `concrete-evidence-column-cages` as class **`tie`** in batch `sam_stirrups`.

The outline covers the whole stirrup including its bent ends; add `hook_90` / `hook_135` on top by hand where the hook is clear (see `team/diego/LABELLING.md`).

**Not used:** Column Tie Detection (its "Tie" boxes are tie-wire knots at crossings, not stirrups) and SteelBindings (boxes around whole zones of stirrups, not one per stirrup).

In [ ]:
!pip -q install ultralytics==8.4.216 roboflow

In [ ]:
%%writefile ce_boxes.py
"""Boxes -> polygon outlines with Segment Anything (SAM 2.1), for Roboflow box-only sets.

A stirrup is a closed loop: its mask has a hole. YOLO-seg and Roboflow polygons
can't have holes, so each hole is joined to the outer edge by a thin cut
("keyhole"), giving one polygon that leaves the inside of the loop unlabelled.
"""
from __future__ import annotations

import cv2
import numpy as np


def _nearest(a: np.ndarray, b: np.ndarray) -> tuple[int, int]:
    """Indices (i in a, j in b) of the closest pair of points."""
    d = ((a[:, None, :].astype(np.float32) - b[None, :, :]) ** 2).sum(-1)
    i, j = np.unravel_index(int(d.argmin()), d.shape)
    return int(i), int(j)


def mask_to_polygon(mask: np.ndarray, min_area: float = 50.0, eps: float = 1.0) -> np.ndarray | None:
    """Largest component of a binary mask -> one polygon (N, 2) px, holes keyholed in."""
    m = (mask > 0.5).astype(np.uint8)
    contours, hier = cv2.findContours(m, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    if not contours or hier is None:
        return None
    hier = hier[0]
    outers = [i for i, h in enumerate(hier) if h[3] < 0]
    o = max(outers, key=lambda i: cv2.contourArea(contours[i]))
    if cv2.contourArea(contours[o]) < min_area:
        return None
    poly = cv2.approxPolyDP(contours[o], eps, True)[:, 0, :]
    holes = [i for i, h in enumerate(hier) if h[3] == o and cv2.contourArea(contours[i]) >= min_area]
    for h in sorted(holes, key=lambda i: -cv2.contourArea(contours[i])):
        hole = cv2.approxPolyDP(contours[h], eps, True)[:, 0, :]
        i, j = _nearest(poly, hole)
        hole = np.roll(hole, -j, axis=0)
        # walk the outer edge to i, jump in, go round the hole, come back out
        poly = np.concatenate([poly[: i + 1], hole, hole[:1], poly[i:]], axis=0)
    return poly


def fill_ratio(mask: np.ndarray, box) -> float:
    x1, y1, x2, y2 = (int(round(v)) for v in box)
    roi = mask[max(0, y1):y2, max(0, x1):x2]
    return float((roi > 0.5).mean()) if roi.size else 0.0


def yolo_line(cls: int, poly: np.ndarray, w: int, h: int) -> str:
    xy = (poly / np.array([w, h], dtype=np.float32)).clip(0, 1).reshape(-1)
    return f"{cls} " + " ".join(f"{v:.6f}" for v in xy)

In [ ]:
# 1. Download the box set (key asked with getpass: never paste it into a file)
from getpass import getpass
from roboflow import Roboflow
WORKSPACE = 'diegos-workspace-puwbt'
BOX_PROJECT, BOX_VERSION = 'stirrup-recognition-dataset-w9jig', 1
TARGET_PROJECT = 'concrete-evidence-column-cages-ft8gc'
rf = Roboflow(api_key=getpass('Roboflow API key: '))
ws = rf.workspace(WORKSPACE)
ws.project(BOX_PROJECT).version(BOX_VERSION).download('yolov8', location='data/stirrups_boxes', overwrite=True)

In [ ]:
# 2. SAM on every box -> keyhole polygon, class tie
import shutil
from pathlib import Path
import cv2
from ultralytics import SAM
from ce_boxes import mask_to_polygon, fill_ratio, yolo_line

SRC, OUT = Path('data/stirrups_boxes'), Path('data/stirrups_sam')
PAD = 0.03            # grow each box a little so the hook tips stay inside
FILL = (0.05, 0.75)   # a loop's mask covers part of its box, never all of it
sam = SAM('sam2.1_b.pt')
shutil.rmtree(OUT, ignore_errors=True)
stats = {'images': 0, 'boxes': 0, 'kept': 0, 'dropped': 0}
review = []
for split in ('train', 'valid', 'test'):
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)
    for img in sorted((SRC / split / 'images').glob('*')):
        lab = SRC / split / 'labels' / (img.stem + '.txt')
        if not lab.exists():
            continue
        im = cv2.imread(str(img)); h, w = im.shape[:2]
        boxes = []
        for line in lab.read_text().split('\n'):
            p = line.split()
            if len(p) != 5:
                continue
            cx, cy, bw, bh = (float(v) for v in p[1:])
            bw, bh = bw * (1 + 2 * PAD), bh * (1 + 2 * PAD)
            boxes.append([max(0, (cx - bw / 2) * w), max(0, (cy - bh / 2) * h),
                          min(w, (cx + bw / 2) * w), min(h, (cy + bh / 2) * h)])
        if not boxes:
            continue
        stats['images'] += 1; stats['boxes'] += len(boxes)
        res = sam(str(img), bboxes=boxes, verbose=False)[0]
        lines, polys = [], []
        for mk, b in zip(res.masks.data.cpu().numpy(), boxes):
            f = fill_ratio(mk, b)
            poly = mask_to_polygon(mk) if FILL[0] < f < FILL[1] else None
            if poly is None or len(poly) < 6:
                stats['dropped'] += 1
                continue
            stats['kept'] += 1; polys.append(poly)
            lines.append(yolo_line(0, poly, w, h))   # class 0 = tie in data.yaml below
        if lines:
            shutil.copy(img, OUT / split / 'images' / img.name)
            (OUT / split / 'labels' / (img.stem + '.txt')).write_text('\n'.join(lines) + '\n')
            if len(review) < 24:
                review.append((img, polys))
(OUT / 'data.yaml').write_text('names:\n- tie\nnc: 1\ntrain: train/images\nval: valid/images\ntest: test/images\n')
print(stats)

In [ ]:
# 3. Check by eye: green = what gets uploaded as tie. Re-run with other PAD / FILL if it looks wrong.
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 6, figsize=(24, 14))
for ax, (img, polys) in zip(axes.flat, review):
    im = cv2.cvtColor(cv2.imread(str(img)), cv2.COLOR_BGR2RGB)
    fill = np.zeros(im.shape[:2], np.uint8)
    for p in polys:
        cv2.fillPoly(fill, [p.astype(np.int32)], 1)
    im[fill > 0] = (0.45 * im[fill > 0] + 0.55 * np.array([0, 255, 0])).astype(np.uint8)
    ax.imshow(im); ax.set_title(img.name, fontsize=8); ax.axis('off')
for ax in axes.flat[len(review):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 4. Upload to our project as class tie (batch sam_stirrups). Only run once the grid looks right.
ws.upload_dataset(str(OUT), TARGET_PROJECT, num_workers=8, project_type='instance-segmentation',
                  batch_name='sam_stirrups')
print('Uploaded. In Roboflow: Annotate -> sam_stirrups -> check a few, add hook_90 / hook_135, Add to dataset.')

Credits: *stirrup-recognition-dataset*, jiawens Workspace, Roboflow Universe, CC BY 4.0. SAM 2.1: Ravi et al. (2024), Apache-2.0, via Ultralytics (AGPL-3.0).